# Asignación de valores

In [1]:
import numpy as np
from datetime import datetime


# ----------------------------
# Frecuencia de conmutación
# ----------------------------
fsw = 50e3
w = 2 * np.pi * fsw
T = 1 / fsw

# ----------------------------
# Voltajes de puertos en lado DC
# ----------------------------
Vdc1 = 210
Vdc2 = 173

# ----------------------------
# Relación de vueltas MFT
# ----------------------------
mx = 1.393  # Parámetro diseño
a12parcial = mx * Vdc1 / Vdc2   # n1/n2

# ----------------------------
# Número de vueltas
# ----------------------------
n1 = 9
n2 = 4

# Relación rectificada
a12 = n1 / n2
N = a12

# ----------------------------
# Valor inductancia auxiliar
# ----------------------------
px = -1.9*(mx)**4 + 12.6*(mx)**3 - 30.9*(mx)**2 + 34.3*(mx) - 14.07

Pmax = 2500

Leq = (px * Vdc1 * Vdc1) / (2 * np.pi * fsw * Pmax)

L12roundu = round(Leq * 1e6)
print(f"L12roundu = {L12roundu} uH")

L12 = L12roundu*1e-6 

L12roundu = 37 uH


# Optimización: Condiciones

In [2]:

h_iSR = 6
h_rest = 48


# Vdc1 fixed
Vdc1 = 210
Vdc1_fijo = Vdc1

# Vdc2 from 130 to 200
Vdc2min = 130 + 7.5 * 0
Vdc2max = 200
pasoVdc2 = 5
dimVdc2 = int((Vdc2max - Vdc2min) / pasoVdc2 + 1)

# Power delivered to port 2 from 0 to 2.5 kW
P2min = 0
P2max = 2500
pasoP2 = 100
dimP2 = int((P2max - P2min) / pasoP2 + 1)

print([Vdc2min, Vdc2max, pasoVdc2, dimVdc2])
print([P2min, P2max, pasoP2, dimP2])

dimTOTALefectiva = dimVdc2 * dimP2
dimTOTALfull = dimVdc2 * dimP2

print("dimTOTALefectiva =", dimTOTALefectiva)
print("dimTOTALfull =", dimTOTALfull)

#DATATPS = np.zeros((dimTOTALfull, 3))


#Loop de optimización


[130.0, 200, 5, 15]
[0, 2500, 100, 26]
dimTOTALefectiva = 390
dimTOTALfull = 390


# Optimización: Operación

$$\boxed{\begin{aligned}
    \min_{x} \quad & F_{\text{OBJ}}(x) \\
    \text{s.t.} \quad
    & P(x) = P_{12}^{*} \\
    & L_b \leq x \leq U_b
\end{aligned}}$$

\begin{aligned}
    F_{\text{OBJ}}(x) = I_{L_{12,\text{RMS}}}^{2}
    = \frac{1}{T} \int_{0}^{T} \left(i_{L_{12}}\right)^{2} \, dt
\end{aligned}

\begin{aligned}
    \mathbf{x} = \begin{bmatrix} x_1 & x_2 & x_3 \end{bmatrix}^{T}
    = \begin{bmatrix} D_1 & D_2 & \varphi \end{bmatrix}^{T}
\end{aligned}

\begin{aligned}
    P(x) = \frac{4 N V_1 V_2}{\pi^3 f_s L_{12}}
    \sum_{k=1,3,5,\dots}^{2h+1}
    \frac{1}{k^3}
    \sin(k \pi D_1)\,\sin(k \pi D_2)\,\sin(k \varphi)
\end{aligned}

\begin{aligned}
    \mathbf{L}_b = \begin{bmatrix} 0 & 0 & -\frac{\pi}{2} \end{bmatrix}
\end{aligned}

\begin{aligned}
    \mathbf{U}_b = \begin{bmatrix} 0.5 & 0.5 & \frac{\pi}{2} \end{bmatrix}
\end{aligned}





In [3]:
import gurobipy as gp
from gurobipy import GRB

Tinicial = datetime.now()
open("results.txt", "w").close()
print("Tinicial =", Tinicial)

comb = 0

# Valores a optimizar
Vdc2_values = np.arange(Vdc2min, Vdc2max + pasoVdc2, pasoVdc2)
P2_values = np.arange(P2min, P2max + pasoP2, pasoP2)


for Vdc2 in Vdc2_values:
    print(f"Vdc2 = {Vdc2}")


    for P12_ref in P2_values:

        print(f"P12_ref = {P12_ref}")

        model = gp.Model("DAB_optimization")

        # ----------------------------
        # DECISION VARIABLES
        # ----------------------------
        x1 = model.addVar(lb=0.05,      ub=0.5,       name="x1")   # d1
        x2 = model.addVar(lb=0.05,      ub=0.5,       name="x2")   # d2
        x3 = model.addVar(lb=-np.pi/2,  ub=np.pi/2,   name="x3")   # phi

        # ----------------------------
        # OBJECTIVE  (iL1SR RMS^2)
        # ----------------------------
        obj          = gp.LinExpr()
        const_factor = (4 / (np.pi * 2 * np.pi * fsw * L12)) ** 2

        for f in range(h_iSR + 1):
            k = 2 * f + 1

            # Linear arguments for trig constraints
            a1 = model.addVar(lb=-k * np.pi * 0.5, ub=k * np.pi * 0.5, name=f"a1_f{f}")
            a2 = model.addVar(lb=-k * np.pi * 0.5, ub=k * np.pi * 0.5, name=f"a2_f{f}")
            a3 = model.addVar(lb=-k * np.pi / 2,   ub=k * np.pi / 2,   name=f"a3_f{f}")
            model.addConstr(a1 == k * np.pi * x1, name=f"ca1_f{f}")
            model.addConstr(a2 == k * np.pi * x2, name=f"ca2_f{f}")
            model.addConstr(a3 == k * x3,          name=f"ca3_f{f}")

            # Trig variables
            s1 = model.addVar(lb=-1, ub=1, name=f"s1_f{f}")
            s2 = model.addVar(lb=-1, ub=1, name=f"s2_f{f}")
            c3 = model.addVar(lb=-1, ub=1, name=f"c3_f{f}")
            model.addGenConstrSin(a1, s1, name=f"sin_a1_f{f}", options="FuncPieces=-1 FuncPieceError=1e-5")
            model.addGenConstrSin(a2, s2, name=f"sin_a2_f{f}", options="FuncPieces=-1 FuncPieceError=1e-5")
            model.addGenConstrCos(a3, c3, name=f"cos_a3_f{f}", options="FuncPieces=-1 FuncPieceError=1e-5")

            # Quadratic cross-terms: need auxiliary vars for products
            s1s1 = model.addVar(lb=0,  ub=1,  name=f"s1s1_f{f}")
            s2s2 = model.addVar(lb=0,  ub=1,  name=f"s2s2_f{f}")
            s1s2 = model.addVar(lb=-1, ub=1,  name=f"s1s2_f{f}")
            s1s2c3 = model.addVar(lb=-1, ub=1, name=f"s1s2c3_f{f}")

            model.addConstr(s1s1   == s1 * s1,       name=f"q_s1s1_f{f}")
            model.addConstr(s2s2   == s2 * s2,       name=f"q_s2s2_f{f}")
            model.addConstr(s1s2   == s1 * s2,       name=f"q_s1s2_f{f}")
            model.addConstr(s1s2c3 == s1s2 * c3,     name=f"q_s1s2c3_f{f}")

            coeff = const_factor * (1 / k**4) * 0.5
            obj  += coeff * (
                Vdc1**2         * s1s1
                + (N * Vdc2)**2   * s2s2
                - 2 * Vdc1 * N * Vdc2 * s1s2c3
            )

        model.setObjective(obj, GRB.MINIMIZE)

        # ----------------------------
        # POWER BALANCE CONSTRAINT
        # ----------------------------
        P12      = gp.LinExpr()
        const_P  = 4 * Vdc1 * N * Vdc2 / (np.pi**3 * fsw * L12)

        for i in range(h_rest + 1):
            k = 2 * i + 1

            b1 = model.addVar(lb=-k * np.pi * 0.5, ub=k * np.pi * 0.5, name=f"b1_i{i}")
            b2 = model.addVar(lb=-k * np.pi * 0.5, ub=k * np.pi * 0.5, name=f"b2_i{i}")
            b3 = model.addVar(lb=-k * np.pi / 2,   ub=k * np.pi / 2,   name=f"b3_i{i}")
            model.addConstr(b1 == k * np.pi * x1, name=f"cb1_i{i}")
            model.addConstr(b2 == k * np.pi * x2, name=f"cb2_i{i}")
            model.addConstr(b3 == k * x3,          name=f"cb3_i{i}")

            s1p = model.addVar(lb=-1, ub=1, name=f"s1p_i{i}")
            s2p = model.addVar(lb=-1, ub=1, name=f"s2p_i{i}")
            s3p = model.addVar(lb=-1, ub=1, name=f"s3p_i{i}")
            model.addGenConstrSin(b1, s1p, name=f"sin_b1_i{i}", options="FuncPieces=-1 FuncPieceError=1e-5")
            model.addGenConstrSin(b2, s2p, name=f"sin_b2_i{i}", options="FuncPieces=-1 FuncPieceError=1e-5")
            model.addGenConstrSin(b3, s3p, name=f"sin_b3_i{i}", options="FuncPieces=-1 FuncPieceError=1e-5")

            # Triple product: s1p * s2p * s3p via two aux vars
            s1ps2p  = model.addVar(lb=-1, ub=1, name=f"s1ps2p_i{i}")
            s1ps2ps3p = model.addVar(lb=-1, ub=1, name=f"s1ps2ps3p_i{i}")
            model.addConstr(s1ps2p    == s1p * s2p,        name=f"q_s1ps2p_i{i}")
            model.addConstr(s1ps2ps3p == s1ps2p * s3p,     name=f"q_triple_i{i}")

            P12 += const_P * (1 / k**3) * s1ps2ps3p

        model.addConstr(P12 == P12_ref, name="power_balance")

        # ----------------------------
        # SOLVER SETTINGS
        # ----------------------------
        model.Params.NonConvex       = 2
        model.Params.TimeLimit        = 300      # seconds
        model.Params.MIPGap           = 1e-4
        model.Params.FuncPieceError   = 1e-5    # global fallback
        model.params.OutputFlag = 0 #No log en consola

        # Warm start
        x1.Start = 0.2
        x2.Start = 0.25
        x3.Start = 0.1

        model.optimize()
        model.write("model.lp")

        # ----------------------------
        # RESULTS
        # ----------------------------
        if model.Status == GRB.OPTIMAL:
            print(f"\nbest_x = [{x1.X:.6f}, {x2.X:.6f}, {x3.X:.6f}]")
            print(f"x3/pi  = {x3.X / np.pi:.6f}")
            print(f"Objective (iL1SR^2) = {model.ObjVal:.6e}")
            print(f"sqrt(obj) = iL1SR_rms = {np.sqrt(model.ObjVal):.6f} A")
        elif model.Status == GRB.INFEASIBLE:
            print("Model is infeasible — computing IIS...")
            model.computeIIS()
            model.write("model.ilp")
        else:
            print(f"Solver status: {model.Status}")
        

        #DATATPS[comb, 0] = x1.X
        #DATATPS[comb, 1] = x2.X
        #DATATPS[comb, 2] = x3.X
        
        with open("results.txt", "a") as file:
            file.write(f"{x1.X:.6f}, {x2.X:.6f}, {x3.X:.6f}\n")


Tfinal = datetime.now()
Ttotal = Tfinal - Tinicial

print("Tfinal =", Tfinal)
print("Ttotal =", Ttotal)
print("comb =", comb)








Tinicial = 2026-06-05 20:26:03.209223
Vdc2 = 130.0
P12_ref = 0
Set parameter Username
Set parameter LicenseID to value 2794452
Academic license - for non-commercial use only - expires 2027-03-18
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.0001
Set parameter FuncPieceError to value 1e-05

best_x = [0.069898, 0.050000, 0.000000]
x3/pi  = 0.000000
Objective (iL1SR^2) = 3.762480e-02
sqrt(obj) = iL1SR_rms = 0.193971 A
P12_ref = 100
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.0001
Set parameter FuncPieceError to value 1e-05

best_x = [0.122263, 0.087788, 0.107775]
x3/pi  = 0.034306
Objective (iL1SR^2) = 1.219665e+00
sqrt(obj) = iL1SR_rms = 1.104384 A
P12_ref = 200
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.0001
Set parameter FuncPieceError to value 1e-05

best_x = [0.173124, 0.124322, 0.152211]
x3/pi  = 0.048450
O

# GUARDAR ARHIVO .C

In [4]:

import numpy as np

DATATPS = np.loadtxt("results.txt", delimiter=",")

print(DATATPS.shape)  # should be (390, 3)


tramas = DATATPS.shape[0]
largoDATA = DATATPS.shape[0] * DATATPS.shape[1]

with open('DAB_210_130a200_2p5kw_zvs_2.c', 'w') as f:

    # Write header parameters
    f.write(f'int Vdc2min_tabla  = {Vdc2min} ; \n')
    f.write(f'int Vdc2max_tabla  = {Vdc2max} ; \n')
    f.write(f'float pasoVdc2_tabla = {pasoVdc2:.6g} ; \n')
    f.write(f'int dimVdc2_tabla  = {dimVdc2} ; \n')

    f.write(f'int P2min_tabla  = {P2min} ; \n')
    f.write(f'int P2max_tabla  = {P2max} ; \n')
    f.write(f'float pasoP2_tabla = {pasoP2:.6g} ; \n')
    f.write(f'int dimP2_tabla  = {dimP2} ; \n')

    f.write(f'int tramas  = {tramas} ; \n')
    f.write(f'int largoDATA = {largoDATA} ; \n\n')

    # Start array
    f.write(f'float DATADAB[{largoDATA}] = {{')

    # Flatten row-wise (same as MATLAB loops)
    for row in DATATPS:
        for val in row:
            f.write(f'{val:.6g},')

    # Close array
    f.write('} ; \n')



(390, 3)
